<div align = "center">

# ***Water Consumption by Crops Data Cleaning***

</div>

## ***Importing Libraries***

In [44]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

## ***Data Loading & Overview:***

### 1. **Loading the Dataset**

In [45]:
def load_data(file_path):
    """ load the dataset from csv file and return dataframe """
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"file not found error: {e}")
        return None
    else:
        print("data loaded successfully")
        return df
    finally:
        print("Execution completed.")

file_path = "../Data/Raw Data/Water Consumption by Crops.csv"
df = load_data(file_path)
df.head()

data loaded successfully
Execution completed.


,Crop,Season,Year,Water_Consumption
0,Sugar Beet,Winter,2011,618118.0
1,Sugar Beet,Winter,2012,574576.0
2,Sugar Beet,Winter,2013,863085.0
3,Sugar Beet,Winter,2014,977227.0
4,Sugar Beet,Winter,2015,1020059.0


### 2. ***Data Overview***

In [46]:
# data shape
print(f"Data shape: {df.shape[0]} rows and {df.shape[1]} columns")

#data info & data types
df.info()

Data shape: 494 rows and 4 columns
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 494 entries, 0 to 493
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Crop               494 non-null    object 
 1   Season             494 non-null    object 
 2   Year               494 non-null    int64  
 3   Water_Consumption  469 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 15.6+ KB


In [47]:
numeric_cols = ['Water_Consumption']

print("Statistical Analysis:")
df[numeric_cols].describe().round(3).T

Statistical Analysis:


,count,mean,std,min,25%,50%,75%,max
Water_Consumption,469.0,934179.703,1853113.546,0.0,9477.1,82479.0,863085.0,9043052.0


In [48]:
print("\nStatistical Summary for Categorical Variables:")
df.describe(include=object).T


Statistical Summary for Categorical Variables:


,count,unique,top,freq
Crop,494,26,vegetables,39
Season,494,3,Winter,195


## ***Data Quality Check & Cleaning***

### 1. ***Data Quality Check***

#### **Missing Values**

In [49]:
missing_values = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage': (df.isnull().sum() / len(df)) * 100})

missing_values

,Missing_Count,Missing_Percentage
Crop,0,0.000000
Season,0,0.000000
Year,0,0.000000
Water_Consumption,25,5.060729


In [50]:
missing_data = df[df['Water_Consumption'].isnull()]

missing_data.sort_values(
    by=['Crop', 'Season', 'Year'])

,Crop,Season,Year,Water_Consumption
338,Alfalfa,Summer,2011,NaN
339,Alfalfa,Summer,2012,NaN
340,Alfalfa,Summer,2013,NaN
341,Alfalfa,Summer,2014,NaN
342,Alfalfa,Summer,2015,NaN
343,Alfalfa,Summer,2016,NaN
344,Alfalfa,Summer,2017,NaN
345,Alfalfa,Summer,2018,NaN
481,Green Fodder,Nile season,2011,NaN
482,Green Fodder,Nile season,2012,NaN


In [51]:
missing_data.groupby(
    ['Crop', 'Season']).agg(
    Missing_Records=('Year', 'count'),
    Missing_Years=('Year', lambda x: sorted(x.tolist()))).reset_index()

,Crop,Season,Missing_Records,Missing_Years
0,Alfalfa,Summer,8,"[2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018]"
1,Green Fodder,Nile season,4,"[2011, 2012, 2013, 2014]"
2,"Medical, Aromatic Plant",Nile season,4,"[2011, 2012, 2013, 2014]"
3,Other varieties,Summer,1,[2019]
4,Other varieties,Winter,4,"[2019, 2021, 2022, 2023]"
5,Sesame,Nile season,4,"[2011, 2012, 2013, 2014]"


In [52]:
for crop in missing_data['Crop'].unique():

    print(f"\nCrop: {crop}")

    display(
        df[df['Crop'] == crop]
        .groupby('Season')
        .agg(Total_Records=('Year', 'count'),
            Available_Values=('Water_Consumption', 'count'),
            Mean_Consumption=('Water_Consumption', 'mean')))


Crop: Other varieties


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Summer,13,12,112574.758333
Winter,13,9,1754.000000



Crop: Alfalfa


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Summer,13,5,22796.0



Crop: Medical, Aromatic Plant


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Nile season,13,9,4724.888889
Summer,13,13,61503.838462
Winter,13,13,72963.846154



Crop: Sesame


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Nile season,13,9,4074.555556
Summer,13,13,133945.507692



Crop: Green Fodder


,Total_Records,Available_Values,Mean_Consumption
Season,,,
Nile season,13,9,3.601322e+04
Summer,13,13,1.060641e+06


#### **Duplicated Rows**

In [53]:
print("Full duplicates:", df.duplicated().sum())
print("Duplicate business keys:", 
    df.duplicated(
        subset=['Crop','Year', 'Season']).sum())

Full duplicates: 0
Duplicate business keys: 0


In [54]:
df.Season.unique()

array(['Winter', 'Summer', 'Nile season'], dtype=object)

In [55]:
df.Crop.unique()

array(['Sugar Beet', 'Wheat', 'Barley', 'Flax', 'vegetables',
       'Medical, Aromatic Plant', 'Beans', 'Lupine', 'Onion', 'Garlic',
       'Fenugreek', 'Lentil', 'Chickpeas', 'Clover', 'Other varieties',
       'Sunflower', 'Sugar Cane', 'Rice', 'Cotton', 'Sorghum', 'Maize',
       'Sesame', 'Green Fodder', 'Alfalfa', 'Peanuts', 'Soybean'],
      dtype=object)

In [56]:
print("\nNegative Values:")
print((df[numeric_cols] < 0).sum())


Negative Values:
Water_Consumption    0
dtype: int64


### 2. ***Data Cleaning***

In [57]:
df_cleaned = df.copy()

In [58]:
df_cleaned.columns = (
    df_cleaned.columns.str.strip()
    .str.replace(" ","_", regex=False)
    .str.replace("-","_", regex=False)
    .str.replace("/","_", regex=False)
    .str.lower())

print(df_cleaned.columns)

Index(['crop', 'season', 'year', 'water_consumption'], dtype='object')


In [59]:
df_cleaned.to_csv('../Data/Clean Data/Water_Consumption_Crops_clean.csv', index=False, encoding='utf-8-sig')

print("CSV file saved successfully!")
print("Shape:", df_cleaned.shape)

CSV file saved successfully!
Shape: (494, 4)
